In [ ]:
!nvidia-smi

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentencepiece

In [ ]:
from huggingface_hub import notebook_login; notebook_login()

In [ ]:
import os
os.makedirs("prompts", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

from google.colab import files
uploaded = files.upload()
for name in uploaded:
    os.replace(name, f"prompts/{name}")

print()
for f in sorted(os.listdir("prompts")):
    n = sum(1 for _ in open(f"prompts/{f}", encoding="utf-8"))
    print(f"  {f}  ({n} prompts)")

In [ ]:
MODELS = {
    "krikri":  "ilsp/Llama-Krikri-8B-Instruct",
    "meltemi": "ilsp/Meltemi-7B-Instruct-v1.5",
    "llama":   "meta-llama/Llama-3.1-8B-Instruct",
}

LOAD_IN_4BIT = True      # True = fits free T4. Keep the same for all three models.
MAX_NEW_TOKENS = 8       # We only want one letter back.
SYSTEM_PROMPT = None     # e.g. "Απαντάς στα ελληνικά." — or leave as None.

print("4-bit:", LOAD_IN_4BIT, "| system prompt:", SYSTEM_PROMPT)

In [ ]:
import json
import re
import gc
import torch

from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)


GREEK_LETTERS = ["α", "β", "γ", "δ", "ε"]

LATIN_TO_GREEK = {
    "a": "α",
    "b": "β",
    "c": "γ",
    "d": "δ",
    "e": "ε",
}


def load_model(hf_id):
    tok = AutoTokenizer.from_pretrained(hf_id)

    kwargs = {
        "device_map": "auto",
        "torch_dtype": torch.float16,
    }

    if LOAD_IN_4BIT:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
        )

    model = AutoModelForCausalLM.from_pretrained(hf_id, **kwargs)
    model.eval()

    return tok, model


def build_chat(tok, prompt):
    """Wrap the plain prompt in whatever chat format this model expects."""

    messages = []

    if SYSTEM_PROMPT:
        messages.append(
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            }
        )

    messages.append(
        {
            "role": "user",
            "content": prompt,
        }
    )

    try:
        return tok.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    except Exception:
        # Model refuses a system role: fold it into the user turn instead.
        merged = (
            SYSTEM_PROMPT + "\n\n" + prompt
            if SYSTEM_PROMPT
            else prompt
        )

        return tok.apply_chat_template(
            [
                {
                    "role": "user",
                    "content": merged,
                }
            ],
            tokenize=False,
            add_generation_prompt=True,
        )


def parse_letter(text, n_options=4):
    """
    Pull a single answer letter out of the model's reply.
    Returns None if it did not give one.

    Reads word by word, not character by character, so that the
    capital Α of a reply such as "Απάντηση: β" is not taken as α.
    """

    valid = GREEK_LETTERS[:n_options]

    # words only: "(β)" gives "β", "Απάντηση: β" gives "Απάντηση", "β"
    for word in re.findall(r"[^\W\d_]+", text):
        low = word.lower()

        if low in valid:
            return low

        if (
            low in LATIN_TO_GREEK
            and LATIN_TO_GREEK[low] in valid
        ):
            return LATIN_TO_GREEK[low]

    return None


def run(model_key, prompt_file):
    hf_id = MODELS[model_key]
    stem = prompt_file.replace(".jsonl", "")
    out_path = f"outputs/{stem}_{model_key}.jsonl"

    with open(
        f"prompts/{prompt_file}",
        encoding="utf-8",
    ) as fh:
        items = [json.loads(line) for line in fh]

    # Resume: skip anything already done.
    done = set()

    if os.path.exists(out_path):
        with open(out_path, encoding="utf-8") as fh:
            done = {
                json.loads(line)["item_id"]
                for line in fh
                if line.strip()
            }

        items = [
            item
            for item in items
            if item["item_id"] not in done
        ]

        print(
            f"  resuming — {len(done)} already done, "
            f"{len(items)} to go"
        )

    if not items:
        print(f"  {out_path} already complete")
        return

    tok, model = load_model(hf_id)

    with open(out_path, "a", encoding="utf-8") as fh:
        for item in tqdm(
            items,
            desc=f"{model_key} / {stem}",
        ):
            chat = build_chat(tok, item["prompt"])

            enc = tok(
                chat,
                return_tensors="pt",
            ).to(model.device)

            with torch.no_grad():
                gen = model.generate(
                    **enc,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=False,
                    pad_token_id=tok.eos_token_id,
                )

            prompt_length = enc["input_ids"].shape[1]

            raw = tok.decode(
                gen[0][prompt_length:],
                skip_special_tokens=True,
            )

            n_opts = len(item["option_positions"])

            result = {
                "item_id": item["item_id"],
                "task": item["task"],
                "condition": item["condition"],
                "model": model_key,
                "raw_output": raw,
                "parsed_letter": parse_letter(raw, n_opts),
                "correct_answer_position": (
                    item["correct_answer_position"]
                ),
            }

            fh.write(
                json.dumps(
                    result,
                    ensure_ascii=False,
                )
                + "\n"
            )
            fh.flush()

    del model, tok
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f"  saved -> {out_path}")


print("helpers ready")

In [ ]:
PROMPT_FILES = sorted(
    f for f in os.listdir("prompts")
    if f.endswith(".jsonl")
)

print(PROMPT_FILES)

In [ ]:
for f in PROMPT_FILES:    run("krikri", f)

In [ ]:
for f in PROMPT_FILES:    run("meltemi", f)

In [ ]:
for f in PROMPT_FILES:    run("llama", f)

In [ ]:
from collections import Counter
import json
import os

for fname in sorted(os.listdir("outputs")):
    path = os.path.join("outputs", fname)

    if not os.path.isfile(path):
        continue

    with open(path, encoding="utf-8") as fh:
        rows = [json.loads(line) for line in fh]

    bad = sum(
        1
        for r in rows
        if r["parsed_letter"] is None
    )

    spread = Counter(
        r["parsed_letter"]
        for r in rows
        if r["parsed_letter"]
    )

    print(fname)
    print(
        f"   n={len(rows)}   "
        f"format violations={bad}   "
        f"answers={dict(sorted(spread.items()))}"
    )

    for r in rows[:2]:
        print(
            f"   sample: {r['raw_output']!r}  -> {r['parsed_letter']}"
        )

    print()

In [ ]:
!zip -qr model_outputs.zip outputs

from google.colab import files

files.download("model_outputs.zip")